# Lacune Segmentation Inference Visualization

This standalone notebook loads trained Experiment A checkpoints, runs inference on one Task3 subject, and visualizes prediction boundaries. It is intended for qualitative review rather than training or statistical evaluation.

The notebook expects the repository layout below by default:

```text
.
├── Model Weight/
│   ├── A0 baseline CV/
│   ├── A2 DM Seg + Reg/
│   ├── A3 DM Uncertainty Weighting/
│   ├── A4a Deep Supervision (Seg + DM)/
│   ├── A4b Deep Supervision (DM)/
│   └── A4c Deep Supervision (Seg)/
└── inference_visualization.ipynb
```

Set `DATA_ROOT`, `WEIGHT_ROOT`, `FOLD_ID`, and `SUBJECT_ID` in the next cell.

In [ ]:
from __future__ import annotations

import gc
import math
import os
import random
from collections import OrderedDict
from pathlib import Path

import cc3d
import ipywidgets as widgets
import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import display
from matplotlib.colors import to_rgba
from monai.data import PersistentDataset, list_data_collate
from monai.inferers import sliding_window_inference
from monai.networks.nets import SwinUNETR
from monai.transforms import (
    Compose,
    CropForegroundd,
    EnsureChannelFirstd,
    EnsureTyped,
    LoadImaged,
    NormalizeIntensityd,
    Orientationd,
    Spacingd,
)
from monai.utils import set_determinism
from sklearn.model_selection import StratifiedKFold
from torch.utils.data import DataLoader

seed = 24
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)
set_determinism(seed=seed)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.cuda.empty_cache()
print(f"Using device: {DEVICE}")

In [ ]:
PROJECT_ROOT = Path.cwd()
DATA_ROOT = Path(os.environ.get("LACUNE_DATA_ROOT", PROJECT_ROOT / "data" / "Task3"))
WEIGHT_ROOT = Path(os.environ.get("LACUNE_WEIGHT_PATH", PROJECT_ROOT / "Model Weight"))
CACHE_DIR = Path(os.environ.get("LACUNE_CACHE_PATH", PROJECT_ROOT / ".cache" / "visualization"))

FOLD_ID = 1
SUBJECT_ID = "sub-206"
BASE_MODALITY = "flair"
VARIANTS_TO_SHOW = ["A0", "A1", "A2", "A3", "A4", "A4c"]

ROI_SIZE = (128, 128, 128)
SPACING = (1.0, 1.0, 1.0)
OVERLAP = 0.6
SW_BATCH_SIZE = 1
PRED_THRESHOLD = 0.5
MIN_VOX_PRED = 5
CONNECTIVITY = 26
MATCH_DISTANCE_VOX = 5.0

VARIANT_DIRS = OrderedDict({
    "A0": WEIGHT_ROOT / "A0 baseline CV",
    "A1": WEIGHT_ROOT / "A2 DM Seg + Reg",
    "A2": WEIGHT_ROOT / "A3 DM Uncertainty Weighting",
    "A3": WEIGHT_ROOT / "A4a Deep Supervision (Seg + DM)",
    "A4": WEIGHT_ROOT / "A4b Deep Supervision (DM)",
    "A4c": WEIGHT_ROOT / "A4c Deep Supervision (Seg)",
})

VARIANT_CFG = {
    "A0": {"out_channels": 2, "is_ds": False},
    "A1": {"out_channels": 3, "is_ds": False},
    "A2": {"out_channels": 3, "is_ds": False},
    "A3": {"out_channels": 3, "is_ds": True},
    "A4": {"out_channels": 3, "is_ds": True},
    "A4c": {"out_channels": 2, "is_ds": True},
}

print(f"DATA_ROOT: {DATA_ROOT}")
print(f"WEIGHT_ROOT: {WEIGHT_ROOT}")
print(f"CACHE_DIR: {CACHE_DIR}")

In [ ]:
def discover_subjects(data_root: Path) -> list[dict]:
    subjects = []
    for subject_dir in sorted(data_root.glob("sub-*")):
        subject_id = subject_dir.name
        subject_num = int(subject_id.split("-")[-1])
        rater = 2 if 101 <= subject_num <= 106 else 4

        flair = list(subject_dir.glob(f"{subject_id}_space-T1_desc-masked_FLAIR.nii*"))
        t1 = list(subject_dir.glob(f"{subject_id}_space-T1_desc-masked_T1.nii*"))
        t2 = list(subject_dir.glob(f"{subject_id}_space-T1_desc-masked_T2.nii*"))
        mask = list(subject_dir.glob(f"{subject_id}_space-T1_desc-Rater{rater}_Lacunes.nii*"))
        if not (len(flair) == len(t1) == len(t2) == len(mask) == 1):
            continue

        has_lacune = bool((nib.load(mask[0]).get_fdata() > 0).any())
        subjects.append({
            "sid": subject_id,
            "flair": str(flair[0]),
            "t1": str(t1[0]),
            "t2": str(t2[0]),
            "mask": str(mask[0]),
            "label": int(has_lacune),
        })
    return subjects


def create_cv_folds(subjects: list[dict], num_folds: int = 5, seed: int = 24) -> list[dict]:
    labels = np.array([subject["label"] for subject in subjects])
    splitter = StratifiedKFold(n_splits=num_folds, shuffle=True, random_state=seed)
    folds = []
    for fold_id, (train_idx, val_idx) in enumerate(splitter.split(np.zeros(len(labels)), labels), 1):
        val_subjects = [subjects[idx] for idx in val_idx]
        folds.append({"fold_id": fold_id, "val": val_subjects})
    return folds


image_keys = ["flair", "t1", "t2"]
all_keys = image_keys + ["mask"]
val_transforms = Compose([
    LoadImaged(keys=all_keys),
    EnsureChannelFirstd(keys=all_keys),
    Orientationd(keys=all_keys, axcodes="RAS"),
    Spacingd(
        keys=all_keys,
        pixdim=SPACING,
        mode=("trilinear", "trilinear", "trilinear", "nearest"),
        align_corners=True,
    ),
    NormalizeIntensityd(keys=image_keys),
    CropForegroundd(keys=all_keys, source_key="flair", margin=10, allow_smaller=True),
    EnsureTyped(keys=all_keys),
])


def get_subject_batch(subject_id: str, fold_id: int):
    subjects = discover_subjects(DATA_ROOT)
    if not subjects:
        raise RuntimeError(f"No Task3 subjects were found under {DATA_ROOT}")
    folds = create_cv_folds(subjects, num_folds=5, seed=seed)
    fold = folds[fold_id - 1]
    val_subjects = fold["val"]
    matching = [subject for subject in val_subjects if subject["sid"] == subject_id]
    if not matching:
        val_ids = [subject["sid"] for subject in val_subjects]
        raise ValueError(f"{subject_id} is not in fold {fold_id}. Available validation subjects: {val_ids}")

    dataset = PersistentDataset(data=matching, transform=val_transforms, cache_dir=str(CACHE_DIR / "val"))
    loader = DataLoader(dataset, batch_size=1, shuffle=False, num_workers=0, collate_fn=list_data_collate)
    return next(iter(loader))


subjects = discover_subjects(DATA_ROOT)
print(f"Discovered subjects: {len(subjects)}")
if subjects:
    folds = create_cv_folds(subjects, num_folds=5, seed=seed)
    print(f"Fold {FOLD_ID} validation subjects: {[subject['sid'] for subject in folds[FOLD_ID - 1]['val']]}")

In [ ]:
class SwinUNETRDS(nn.Module):
    def __init__(self, in_channels: int = 3, out_channels: int = 3, feature_size: int = 48):
        super().__init__()
        self.base = SwinUNETR(
            in_channels=in_channels,
            out_channels=out_channels,
            feature_size=feature_size,
            use_checkpoint=True,
            use_v2=True,
        )

    def load_from(self, weights):
        self.base.load_from(weights)

    def forward(self, x):
        return self.base(x)


def unwrap_state_dict(raw):
    state = raw["state_dict"] if isinstance(raw, dict) and "state_dict" in raw else raw
    if state:
        first_key = next(iter(state.keys()))
        if first_key.startswith("module."):
            state = {key[len("module."):]: value for key, value in state.items()}
    return state


def is_deep_supervision_state(state_dict) -> bool:
    return bool(state_dict) and next(iter(state_dict.keys())).startswith("base.")


def resolve_checkpoint(variant_key: str, fold_id: int) -> Path:
    checkpoint_dir = VARIANT_DIRS[variant_key]
    checkpoint = checkpoint_dir / f"fold{fold_id}_best.pth"
    if checkpoint.exists():
        return checkpoint
    candidates = sorted(checkpoint_dir.glob(f"*fold{fold_id}*best*.pth"))
    if candidates:
        return candidates[0]
    raise FileNotFoundError(f"No checkpoint found for {variant_key}, fold {fold_id}: {checkpoint_dir}")


def load_variant_model(variant_key: str, fold_id: int):
    cfg = VARIANT_CFG[variant_key]
    checkpoint = resolve_checkpoint(variant_key, fold_id)
    raw = torch.load(checkpoint, map_location=DEVICE)
    state = unwrap_state_dict(raw)
    use_ds = is_deep_supervision_state(state) or bool(cfg.get("is_ds", False))
    out_channels = int(cfg["out_channels"])

    if use_ds:
        model = SwinUNETRDS(in_channels=3, out_channels=out_channels, feature_size=48)
        model.load_state_dict(state, strict=True)
        predictor = model.base
    else:
        model = SwinUNETR(
            in_channels=3,
            out_channels=out_channels,
            feature_size=48,
            use_checkpoint=True,
            use_v2=True,
        )
        model.load_state_dict(state, strict=True)
        predictor = model

    model.to(DEVICE).eval()
    return model, predictor, checkpoint


def connected_components(mask: np.ndarray, min_voxels: int = 1) -> np.ndarray:
    labels = cc3d.connected_components(mask.astype(np.uint8), connectivity=CONNECTIVITY)
    if min_voxels > 1 and labels.max() > 0:
        component_ids, counts = np.unique(labels, return_counts=True)
        small = component_ids[(counts < min_voxels) & (component_ids != 0)]
        if small.size:
            labels[np.isin(labels, small)] = 0
    return labels.astype(np.int32)


def component_centroids(labels: np.ndarray):
    ids = np.setdiff1d(np.unique(labels), 0)
    if ids.size == 0:
        return np.empty((0, 3), dtype=float), ids
    centroids = [np.argwhere(labels == component_id).mean(axis=0) for component_id in ids]
    return np.asarray(centroids, dtype=float), ids


def match_prediction_components(pred_labels: np.ndarray, gt_labels: np.ndarray, distance_threshold: float):
    pred_centroids, pred_ids = component_centroids(pred_labels)
    gt_centroids, gt_ids = component_centroids(gt_labels)
    tp_ids, fp_ids, matched_gt = set(), set(), set()

    for idx, pred_id in enumerate(pred_ids):
        if gt_centroids.size == 0:
            fp_ids.add(int(pred_id))
            continue
        distances = np.linalg.norm(pred_centroids[idx] - gt_centroids, axis=1)
        best_idx = int(distances.argmin())
        gt_id = int(gt_ids[best_idx])
        if distances[best_idx] < distance_threshold and gt_id not in matched_gt:
            tp_ids.add(int(pred_id))
            matched_gt.add(gt_id)
        else:
            fp_ids.add(int(pred_id))
    return tp_ids, fp_ids


def predict_variant(variant_key: str, fold_id: int, batch):
    model, predictor, checkpoint = load_variant_model(variant_key, fold_id)
    inputs = torch.cat([batch[key].to(DEVICE) for key in image_keys], dim=1)
    with torch.no_grad(), torch.autocast(device_type=DEVICE.type, enabled=(DEVICE.type == "cuda")):
        logits = sliding_window_inference(
            inputs,
            roi_size=ROI_SIZE,
            sw_batch_size=SW_BATCH_SIZE,
            overlap=OVERLAP,
            predictor=predictor,
        )
    seg_logits = logits[:, :2]
    probability = torch.softmax(seg_logits, dim=1)[0, 1].float().cpu().numpy()
    mask = (probability > PRED_THRESHOLD).astype(np.uint8)
    labels = connected_components(mask, min_voxels=MIN_VOX_PRED)
    del model
    torch.cuda.empty_cache()
    gc.collect()
    return {"probability": probability, "mask": mask, "labels": labels, "checkpoint": checkpoint}


batch = get_subject_batch(SUBJECT_ID, FOLD_ID)
base_volume = batch[BASE_MODALITY].cpu().numpy().squeeze()
gt_mask = batch["mask"].cpu().numpy().squeeze().astype(np.uint8)
gt_labels = connected_components(gt_mask, min_voxels=1)

predictions = {}
for variant in VARIANTS_TO_SHOW:
    print(f"Running {variant}...")
    predictions[variant] = predict_variant(variant, FOLD_ID, batch)
    tp_ids, fp_ids = match_prediction_components(predictions[variant]["labels"], gt_labels, MATCH_DISTANCE_VOX)
    predictions[variant]["tp_ids"] = tp_ids
    predictions[variant]["fp_ids"] = fp_ids
    print(f"  checkpoint: {predictions[variant]['checkpoint'].name}")
    print(f"  components: TP={len(tp_ids)}, FP={len(fp_ids)}")

print("Inference complete.")

In [ ]:
def rotate_for_display(slice_2d: np.ndarray) -> np.ndarray:
    return np.rot90(slice_2d, k=3)


def draw_base(ax, volume: np.ndarray, z: int, title: str):
    ax.imshow(rotate_for_display(volume[z]), cmap="gray", origin="lower")
    ax.set_title(title)
    ax.axis("off")


def draw_gt(ax, volume: np.ndarray, mask: np.ndarray, z: int, title: str):
    base = rotate_for_display(volume[z])
    mask_slice = rotate_for_display(mask[z].astype(float))
    ax.imshow(base, cmap="gray", origin="lower")
    overlay = np.zeros((*mask_slice.shape, 4), dtype=float)
    overlay[mask_slice > 0.5] = to_rgba("cyan", alpha=0.12)
    ax.imshow(overlay, origin="lower")
    if mask_slice.max() > 0:
        ax.contour(mask_slice, levels=[0.5], colors=("cyan",), linewidths=1.4, origin="lower")
    ax.set_title(title)
    ax.axis("off")


def draw_prediction(ax, volume: np.ndarray, labels: np.ndarray, z: int, tp_ids: set[int], fp_ids: set[int], title: str):
    ax.imshow(rotate_for_display(volume[z]), cmap="gray", origin="lower")
    for component_id in np.setdiff1d(np.unique(labels), 0):
        component_slice = (labels[z] == component_id).astype(float)
        if component_slice.max() == 0:
            continue
        color = "lime" if int(component_id) in tp_ids else "red"
        ax.contour(
            rotate_for_display(component_slice),
            levels=[0.5],
            colors=(color,),
            linewidths=1.3,
            origin="lower",
        )
    ax.set_title(title)
    ax.axis("off")


def show_static_panel(z: int | None = None, save_path: str | None = None):
    if z is None:
        z = int(base_volume.shape[0] // 2)
    n_variants = len(VARIANTS_TO_SHOW)
    n_cols = 3
    n_rows = int(math.ceil((n_variants + 2) / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4.5 * n_cols, 4.2 * n_rows))
    axes = np.asarray(axes).reshape(-1)

    draw_base(axes[0], base_volume, z, f"{BASE_MODALITY.upper()} slice {z}")
    draw_gt(axes[1], base_volume, gt_mask, z, "Ground truth")
    for idx, variant in enumerate(VARIANTS_TO_SHOW, start=2):
        pred = predictions[variant]
        draw_prediction(
            axes[idx],
            base_volume,
            pred["labels"],
            z,
            pred["tp_ids"],
            pred["fp_ids"],
            variant,
        )
    for ax in axes[n_variants + 2:]:
        ax.axis("off")

    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=200, bbox_inches="tight")
        print(f"Saved figure: {save_path}")
    plt.show()


def show_interactive_viewer():
    slider = widgets.IntSlider(
        min=0,
        max=int(base_volume.shape[0] - 1),
        step=1,
        value=int(base_volume.shape[0] // 2),
        description="slice",
        continuous_update=False,
    )
    output = widgets.Output()

    def redraw(change=None):
        with output:
            output.clear_output(wait=True)
            show_static_panel(z=int(slider.value))

    slider.observe(redraw, names="value")
    display(widgets.VBox([slider, output]))
    redraw()


show_interactive_viewer()

In [ ]:
# Optional: save a static panel for a selected slice.
# show_static_panel(z=base_volume.shape[0] // 2, save_path="visualization_panel.png")